In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd
import folium
from folium.plugins import HeatMap


In [2]:
spark = SparkSession.builder \
    .appName("HeatmapAnalysis") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

print("Spark initialized.")


Spark initialized.


In [3]:
df = spark.read.parquet("hdfs://namenode:9000/data/processed/chicago_crimes_clean.parquet")

print("Loaded records:", df.count())
print(df.columns)


Loaded records: 8422096
['ID', 'Case Number', 'Date', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'Latitude', 'Longitude', 'crime_datetime', 'crime_year', 'crime_month', 'crime_day', 'crime_hour', 'crime_day_of_week', 'time_of_day', 'season', 'location_category', 'location_risk_weight', 'crime_category', 'crime_severity_level', 'crime_severity_weight', 'is_violent', 'is_night', 'was_arrest_made', 'is_domestic_incident', 'is_public_space', 'risk_score', 'Year']


In [4]:
# Select only spatial coordinates from the original Spark DataFrame
# - Latitude / Longitude are used for spatial distribution analysis
# - dropna(): remove rows with missing coordinates
# - sample(False, 0.01): randomly sample 1% of the data to reduce size
#   (original dataset has 8M+ rows, full data would be too slow)
# - seed=42: fixed random seed for reproducibility
pdf = (
    df.select("Latitude", "Longitude")
      .dropna()
      .sample(False, 0.01, seed=42)
      .toPandas()
)

print(pdf.head())
print("Sample size:", len(pdf))


    Latitude  Longitude
0  41.841556 -87.725696
1  41.782639 -87.769232
2  41.778693 -87.683625
3  41.742573 -87.648666
4  41.956026 -87.758029
Sample size: 83234


In [5]:
import folium
from folium.plugins import HeatMap

# Create a Folium map centered on Chicago
# location: approximate coordinates of downtown Chicago
# zoom_start: initial zoom level (11 is suitable for city-wide view)
m = folium.Map(
    location=[41.8781, -87.6298],
    zoom_start=11
)

# Convert coordinates to HeatMap input format:
# [[lat1, lon1], [lat2, lon2], ...]
heat_data = pdf[["Latitude", "Longitude"]].values.tolist()

# Add heatmap layer
# radius: influence radius of each point
# blur: smoothing factor for better visual continuity
HeatMap(
    heat_data,
    radius=7,
    blur=4
).add_to(m)

output_path = "/home/jovyan/notebooks/outputs/crime_heatmap.html"
m.save(output_path)

print("Saved heatmap:", output_path)

Saved heatmap: /home/jovyan/notebooks/outputs/crime_heatmap.html


In [6]:
from IPython.display import IFrame

IFrame("outputs/crime_heatmap.html", width=900, height=600)


In [7]:
spark.stop()